# Phi-4 Multimodal built-in Vision LoRA SFT

## Goal

`microsoft/Phi-4-multimodal-instruct`의 내장 `vision-lora`만 학습해
네 장의 프레임 순서를 생성하도록 만드는 Colab 노트북입니다.

- 기준 실행 흐름: `qwen3-8b-full-finetuning.ipynb`
- 학습 방식: full fine-tuning이 아닌 LoRA SFT
- 데이터 분할: seed 42 MD5-group-safe 80:20 holdout 1회
- 상태: 코드 준비만 완료. 이 파일에는 실행 결과나 학습 체크포인트가 없습니다.
- 안전장치: GPU 확인과 1-batch dry run을 각각 확인해야 실제 학습이 시작됩니다.

공식 소스:
[고정 모델 revision](https://huggingface.co/microsoft/Phi-4-multimodal-instruct/tree/9a490d7b6d2a4e17c70b39b2914ef28e3f2f7c39),
[Transformers Phi-4 Multimodal 문서](https://huggingface.co/docs/transformers/model_doc/phi4_multimodal)


In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
print("공개 Hugging Face 모델을 인증 없이 사용합니다.")


공개 Hugging Face 모델을 인증 없이 사용합니다.


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [3]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
DATA_ROOT = WORKDIR / "snuaichallenge_data"


def find_drive_file(filename):
    direct = DRIVE_ROOT / filename
    if direct.is_file():
        return direct
    matches = list(Path("/content/drive/MyDrive").rglob(filename))
    return matches[0] if matches else None


# 기존 full-train bundle이 train.csv와 38,140개 train 이미지의 입력이다.
if not (DATA_ROOT / "train.csv").is_file() or not (DATA_ROOT / "train").is_dir():
    train_bundle = find_drive_file("snu_colab_full_train_bundle.tar")
    if train_bundle is None:
        raise FileNotFoundError(
            "MyDrive에서 snu_colab_full_train_bundle.tar를 찾지 못했습니다."
        )
    WORKDIR.mkdir(parents=True, exist_ok=True)
    print(f"기존 학습 bundle 압축 해제: {train_bundle}")
    subprocess.run(
        ["tar", "-xf", str(train_bundle), "-C", str(WORKDIR)],
        check=True,
    )
    DRIVE_ROOT = train_bundle.parent

assert (DATA_ROOT / "train.csv").is_file(), DATA_ROOT / "train.csv"
assert (DATA_ROOT / "train").is_dir(), DATA_ROOT / "train"

# Test는 학습/검증에 사용하지 않는다. 제출 셀을 실행할 때만 선택적으로 복구한다.
test_bundle = find_drive_file("snu_colab_test_bundle.tar")
if test_bundle is not None and (
    not (DATA_ROOT / "test.csv").is_file()
    or not (DATA_ROOT / "test").is_dir()
    or not (DATA_ROOT / "sample_submission.csv").is_file()
):
    print(f"기존 test bundle 압축 해제: {test_bundle}")
    subprocess.run(
        ["tar", "-xf", str(test_bundle), "-C", str(WORKDIR)],
        check=True,
    )

DATA_DIR = str(DATA_ROOT)
OUTPUT_DIR = str(DRIVE_ROOT / "snu_phi4_multimodal_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Training data: {DATA_DIR}")
print(f"Persistent output: {OUTPUT_DIR}")


기존 학습 bundle 압축 해제: /content/drive/MyDrive/snu ai/snu_colab_full_train_bundle.tar
기존 test bundle 압축 해제: /content/drive/MyDrive/snu ai/snu_colab_test_bundle.tar
Training data: /content/snu_ai/snuaichallenge_data
Persistent output: /content/drive/MyDrive/snu ai/snu_phi4_multimodal_outputs


## Setup

### 1. Install pinned runtime packages

Phi-4 네이티브 모델/processor와 PEFT adapter 로딩 동작을 고정합니다.
`qwen-vl-utils`는 설치하지 않습니다.


In [5]:
# Colab currently preinstalls torchao 0.10.0, but PEFT 0.19.1 rejects
# installed torchao versions below 0.16.0 even for this plain BF16 LoRA path.
# Phi-4 here is not torchao-quantized, so remove that optional package.
!pip uninstall -y -q torchao
!pip install -q "transformers==5.13.0" "peft==0.19.1" accelerate bitsandbytes safetensors "scikit-learn==1.8.0" "networkx==3.6.1"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 168.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 204.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 52.6 MB/s eta 0:00:00:00:0100:01


In [6]:
import ast
import gc
import hashlib
import json
import os
import random
import re
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import PackageNotFoundError, version as package_version
from itertools import permutations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Phi4MultimodalFeatureExtractor,
    Phi4MultimodalImageProcessor,
    Phi4MultimodalProcessor,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "microsoft/Phi-4-multimodal-instruct"

# 2025-04-17에 Microsoft 저장소에 공개된 네이티브 HF 변환 커밋.
# 원본 모델 공개(2025-03-03) 및 대회 2026-05-31 가중치 컷오프 이전이다.
MODEL_REVISION = "9a490d7b6d2a4e17c70b39b2914ef28e3f2f7c39"
VISION_ADAPTER_SUBFOLDER = "vision-lora"

# 네이티브 변환 커밋에 포함된 기본 template은 text-only message용이다.
# 이 template은 기존 content block 순서를 그대로 순회하며 Phi image token을 삽입한다.
PHI4_CONTENT_CHAT_TEMPLATE = (
    "{%- for message in messages -%}"
    "{{ '<|' + message['role'] + '|>' }}"
    "{%- if message['content'] is string -%}"
    "{{ message['content'] }}"
    "{%- else -%}"
    "{%- for block in message['content'] -%}"
    "{%- if block['type'] == 'image' -%}"
    "{{ image_token }}"
    "{%- elif block['type'] == 'text' -%}"
    "{{ block['text'] }}"
    "{%- endif -%}"
    "{%- endfor -%}"
    "{%- endif -%}"
    "{{ '<|end|>' }}"
    "{%- endfor -%}"
    "{%- if add_generation_prompt -%}"
    "{{ '<|assistant|>' }}"
    "{%- else -%}"
    "{{ eos_token }}"
    "{%- endif -%}"
)

VALIDATION_RATIO = 0.2
MAX_SAMPLES_PER_HASH = 3

# 실제 train 이미지 38,140개는 최대 640x640이고 448-grid crop 수가 최대 4다.
# 기본값 36 대신 4로 고정해 모든 원본 해상도를 보존하면서 activation을 제한한다.
PHI_DYNAMIC_HD = 4

PACKAGE_VERSIONS = {
    package: package_version(package)
    for package in (
        "torch",
        "torchvision",
        "transformers",
        "peft",
        "accelerate",
        "bitsandbytes",
        "safetensors",
        "scikit-learn",
        "networkx",
    )
}
try:
    incompatible_torchao_version = package_version("torchao")
except PackageNotFoundError:
    incompatible_torchao_version = None
assert incompatible_torchao_version is None, (
    "torchao is not used by this BF16 LoRA notebook and must be removed before "
    f"PEFT adapter injection; found {incompatible_torchao_version}. "
    "Rerun the package cell, restart the runtime, and run from the top."
)
print("Runtime package versions:", PACKAGE_VERSIONS)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
train["answer_tuple"] = train["Answer"].apply(
    lambda value: str(ast.literal_eval(value))
)
print(f"Train rows: {len(train):,}")


Runtime package versions: {'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'transformers': '5.13.0', 'peft': '0.19.1', 'accelerate': '1.14.0', 'bitsandbytes': '0.49.2', 'safetensors': '0.8.0', 'scikit-learn': '1.8.0', 'networkx': '3.6.1'}
Train rows: 9,535


## Steps

### 2. Build one MD5-group-safe 80:20 holdout


In [7]:
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'path': path})
    return records

print("Train 이미지 해시 계산 중...")
train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h

hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

hash_to_samples = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}
excluded = sum(1 for h, s in hash_counter.items() if len(s) > MAX_SAMPLES_PER_HASH)
print(f"제외된 해시(빈번 재사용) 수: {excluded}")

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i+1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
print(f"총 {len(components)}개 클러스터, 최대 크기={max(len(c) for c in components)}")

id_to_cluster = {sid: cid for cid, comp in enumerate(components) for sid in comp}
train['cluster_id'] = train['Id'].map(id_to_cluster)

# StratifiedGroupKFold is used only to select one deterministic 20% holdout.
# This is one 80:20 experiment, not five-fold cross-validation.
holdout_partitions = round(1 / VALIDATION_RATIO)
assert holdout_partitions == 5 and abs(VALIDATION_RATIO - 0.2) < 1e-9
splitter = StratifiedGroupKFold(
    n_splits=holdout_partitions, shuffle=True, random_state=SEED
)
train_idx, val_idx = next(
    splitter.split(train, train['answer_tuple'], groups=train['cluster_id'])
)
train['split'] = 'train'
train.loc[val_idx, 'split'] = 'validation'

train_groups = set(train.loc[train_idx, 'cluster_id'])
validation_groups = set(train.loc[val_idx, 'cluster_id'])
assert train_groups.isdisjoint(validation_groups)
assert train.loc[train_idx, 'answer_tuple'].nunique() == 24
assert train.loc[val_idx, 'answer_tuple'].nunique() == 24
print(f"Train: {len(train_idx)}, validation: {len(val_idx)}")
print(f"MD5 clusters crossing the split: {len(train_groups & validation_groups)}")
print("Validation target counts:")
print(train.loc[val_idx, 'answer_tuple'].value_counts().sort_index())
print("Validation No_ordering rows:", int(train.loc[val_idx, 'No_ordering'].sum()))


Train 이미지 해시 계산 중...


  0%|          | 0/38140 [00:00<?, ?it/s]

제외된 해시(빈번 재사용) 수: 2081
총 8147개 클러스터, 최대 크기=14
Train: 7628, validation: 1907
MD5 clusters crossing the split: 0
Validation target counts:
answer_tuple
[1, 2, 3, 4]    296
[1, 2, 4, 3]     71
[1, 3, 2, 4]     72
[1, 3, 4, 2]     69
[1, 4, 2, 3]     69
[1, 4, 3, 2]     71
[2, 1, 3, 4]     68
[2, 1, 4, 3]     73
[2, 3, 1, 4]     61
[2, 3, 4, 1]     73
[2, 4, 1, 3]     70
[2, 4, 3, 1]     69
[3, 1, 2, 4]     72
[3, 1, 4, 2]     67
[3, 2, 1, 4]     72
[3, 2, 4, 1]     68
[3, 4, 1, 2]     77
[3, 4, 2, 1]     70
[4, 1, 2, 3]     63
[4, 1, 3, 2]     67
[4, 2, 1, 3]     71
[4, 2, 3, 1]     71
[4, 3, 1, 2]     74
[4, 3, 2, 1]     73
Name: count, dtype: int64
Validation No_ordering rows: 296


### 3. Keep the Qwen prompt and answer transform unchanged

모델별로 달라지는 것은 이미지 로딩 방식뿐입니다. 힌트, 지시문, content
블록 순서, 정답 역변환은 기준 노트북과 동일합니다.


In [8]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses


In [9]:
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)


def load_rgb_image(image_path):
    with Image.open(image_path) as image:
        return image.convert("RGB")


def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({
            "type": "image",
            "image": load_rgb_image(img_path),
        })
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def format_training_sample(row, img_dir):
    messages = get_prompt_message(row, img_dir)
    answer = ast.literal_eval(row['Answer'])
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target_text}]})
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False


### 4. Inspect and confirm the training GPU

첫 셀은 GPU와 권장 micro-batch를 출력합니다. 실제 GPU/VRAM을 확인한 뒤
다음 셀의 `PHI4_LORA_GPU_CONFIRMED`만 직접 `True`로 바꾸세요.


In [10]:
assert torch.cuda.is_available(), "CUDA GPU가 필요합니다."
assert torch.cuda.is_bf16_supported(), "이 설정은 BF16 지원 GPU가 필요합니다."

gpu = torch.cuda.get_device_properties(0)
GPU_NAME = gpu.name
GPU_VRAM_GB = gpu.total_memory / 1024**3
print(f"GPU: {GPU_NAME}")
print(f"VRAM: {GPU_VRAM_GB:.1f} GiB")
assert GPU_VRAM_GB >= 22, (
    f"감지된 VRAM은 {GPU_VRAM_GB:.1f} GiB입니다. "
    "이 4-image Phi-4 LoRA 설정은 24 GB급 GPU부터 시작하세요."
)

if GPU_VRAM_GB >= 70:
    TRAIN_BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    EVAL_BATCH_SIZE = 4
    INFERENCE_BATCH_SIZE = 4
elif GPU_VRAM_GB >= 40:
    TRAIN_BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 4
    EVAL_BATCH_SIZE = 2
    INFERENCE_BATCH_SIZE = 2
else:
    TRAIN_BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 8
    EVAL_BATCH_SIZE = 1
    INFERENCE_BATCH_SIZE = 1

EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS
assert EFFECTIVE_BATCH_SIZE == 8
print(
    f"권장 설정: train micro-batch={TRAIN_BATCH_SIZE}, "
    f"grad accumulation={GRAD_ACCUM_STEPS}, effective batch={EFFECTIVE_BATCH_SIZE}"
)
print(
    f"Phi image dynamic_hd={PHI_DYNAMIC_HD}, "
    f"validation batch={EVAL_BATCH_SIZE}, inference batch={INFERENCE_BATCH_SIZE}"
)


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GiB
권장 설정: train micro-batch=4, grad accumulation=2, effective batch=8
Phi image dynamic_hd=4, validation batch=4, inference batch=4


In [11]:
# 위 GPU 이름, VRAM, 권장 batch 설정을 실제로 확인한 뒤에만 True로 바꾸세요.
PHI4_LORA_GPU_CONFIRMED = True
assert PHI4_LORA_GPU_CONFIRMED is True, (
    "GPU 확인 전에는 모델을 로드하지 않습니다. "
    "출력된 GPU/VRAM을 확인한 뒤 PHI4_LORA_GPU_CONFIRMED=True로 바꾸세요."
)


In [12]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print(f"GPU 설정 확인: {GPU_NAME} ({GPU_VRAM_GB:.1f} GiB)")
print("BF16 + SDPA + TF32 fallback + local SSD input enabled")


GPU 설정 확인: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
BF16 + SDPA + TF32 fallback + local SSD input enabled


### 5. Load Phi-4 and train only its built-in vision LoRA

별도 `LoraConfig`/`get_peft_model`을 추가하지 않습니다. 공식 체크포인트의
`vision-lora`를 `vision`이라는 이름으로 로드한 뒤 그 A/B 행렬만 FP32로
학습합니다.


In [13]:
assert PHI4_LORA_GPU_CONFIRMED is True


def load_phi4_with_vision_adapter(is_trainable):
    # The pinned native-conversion commit still labels its image processor as
    # `Phi4MMImageProcessor`, so AutoProcessor cannot resolve the native class.
    # Load the three official native components explicitly instead.
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        trust_remote_code=False,
    )
    loaded_image_processor = Phi4MultimodalImageProcessor.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    loaded_audio_processor = Phi4MultimodalFeatureExtractor.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    loaded_processor = Phi4MultimodalProcessor(
        image_processor=loaded_image_processor,
        audio_processor=loaded_audio_processor,
        tokenizer=loaded_tokenizer,
    )
    assert loaded_processor.image_token_id == 200010
    assert loaded_processor.audio_token_id == 200011
    loaded_processor.chat_template = PHI4_CONTENT_CHAT_TEMPLATE
    loaded_processor.tokenizer.padding_side = "right"
    print(
        "Native Phi processor ready:",
        loaded_processor.__class__.__name__,
        loaded_image_processor.__class__.__name__,
        loaded_audio_processor.__class__.__name__,
    )

    loaded_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
        trust_remote_code=False,
        dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    adapter_info = loaded_model.load_adapter(
        MODEL_NAME,
        adapter_name="vision",
        is_trainable=is_trainable,
        device_map="auto",
        adapter_kwargs={
            "subfolder": VISION_ADAPTER_SUBFOLDER,
            "revision": MODEL_REVISION,
        },
    )
    loaded_model.set_adapter("vision")
    return loaded_processor, loaded_model, adapter_info


processor, model, adapter_info = load_phi4_with_vision_adapter(
    is_trainable=True
)

assert model.__class__.__name__ == "Phi4MultimodalForCausalLM", model.__class__
assert processor.__class__.__name__ == "Phi4MultimodalProcessor", processor.__class__
assert processor.image_processor.__class__.__name__ == "Phi4MultimodalImageProcessor"
assert processor.audio_processor.__class__.__name__ == "Phi4MultimodalFeatureExtractor"
assert model.active_adapters() == ["vision"], model.active_adapters()
vision_peft_config = model.peft_config["vision"]
assert vision_peft_config.r == 256, vision_peft_config.r
assert vision_peft_config.lora_alpha == 512, vision_peft_config.lora_alpha
assert vision_peft_config.inference_mode is False

device_map = getattr(model, "hf_device_map", {"": str(model.device)})
device_targets = {str(target).lower() for target in device_map.values()}
assert not any(
    target == "cpu" or "disk" in target for target in device_targets
), f"CPU/disk offload는 학습에 사용하지 않습니다: {device_map}"
assert model.device.type == "cuda", model.device

# 전체 base를 freeze한 뒤 내장 vision adapter A/B만 해동한다.
for _, parameter in model.named_parameters():
    parameter.requires_grad_(False)

trainable_names = []
for name, parameter in model.named_parameters():
    if ("lora_A.vision" in name) or ("lora_B.vision" in name):
        parameter.requires_grad_(True)
        parameter.data = parameter.data.to(torch.float32)
        trainable_names.append(name)

lora_like = [
    name for name, _ in model.named_parameters()
    if "lora" in name.lower()
]
assert trainable_names, (
    "vision LoRA 파라미터를 찾지 못했습니다. "
    f"실제 LoRA 이름 예시: {lora_like[:20]}"
)
assert all(
    ("lora_A.vision" in name) or ("lora_B.vision" in name)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
)

TRAINABLE_PARAM_NAMES = tuple(trainable_names)
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)
print(f"전체 파라미터: {total_params:,}")
print(
    f"학습 파라미터: {trainable_params:,} "
    f"({trainable_params / total_params:.2%})"
)
print(f"학습 tensor 수: {len(TRAINABLE_PARAM_NAMES)}")
print("학습 이름 예시:", list(TRAINABLE_PARAM_NAMES[:8]))

# Transformers 5.13 filters the adapter load report by substring markers
# {"vision", "lora"}. Four already-loaded base audio-projection keys contain
# `vision_speech`, so they are harmless false positives in `missing_keys`.
reported_missing_keys = sorted(getattr(adapter_info, "missing_keys", []))
unexpected_adapter_keys = sorted(getattr(adapter_info, "unexpected_keys", []))
known_base_key_false_positives = {
    "model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.bias",
    "model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.weight",
    "model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.bias",
    "model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.weight",
}
missing_lora_keys = sorted(
    key for key in reported_missing_keys if "lora_" in key.lower()
)
unknown_missing_keys = sorted(
    set(reported_missing_keys) - known_base_key_false_positives
)
assert not missing_lora_keys, missing_lora_keys[:20]
assert not unknown_missing_keys, unknown_missing_keys[:20]
assert not unexpected_adapter_keys, unexpected_adapter_keys[:20]
if reported_missing_keys:
    print(
        "Ignoring known base-key false positives from the adapter report:",
        reported_missing_keys,
    )

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()
model.config.use_cache = False


def apply_phi_chat_template(messages, **kwargs):
    return processor.apply_chat_template(
        messages,
        chat_template=PHI4_CONTENT_CHAT_TEMPLATE,
        image_token=processor.image_token,
        eos_token=processor.tokenizer.eos_token,
        **kwargs,
    )


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/3.54k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json:   0%|          | 0.00/3.47k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/3.91M [00:00<?, ?B/s]

...

adapter_config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

[transformers] Phi4MultimodalForCausalLM LOAD REPORT from: microsoft/Phi-4-multimodal-instruct
Key                                                                      | Status  | 
-------------------------------------------------------------------------+---------+-
model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.bias   | MISSING | 
model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.weight | MISSING | 
model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.bias     | MISSING | 
model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.weight   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


전체 파라미터: 5,113,086,784
학습 파라미터: 369,098,752 (7.22%)
학습 tensor 수: 256
학습 이름 예시: ['model.layers.0.self_attn.o_proj.lora_A.vision.weight', 'model.layers.0.self_attn.o_proj.lora_B.vision.weight', 'model.layers.0.self_attn.qkv_proj.lora_A.vision.weight', 'model.layers.0.self_attn.qkv_proj.lora_B.vision.weight', 'model.layers.0.mlp.gate_up_proj.lora_A.vision.weight', 'model.layers.0.mlp.gate_up_proj.lora_B.vision.weight', 'model.layers.0.mlp.down_proj.lora_A.vision.weight', 'model.layers.0.mlp.down_proj.lora_B.vision.weight']
Ignoring known base-key false positives from the adapter report: ['model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.bias', 'model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.weight', 'model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.bias', 'model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.weight']


## Checks

### 6. Build native Phi batches and inspect the first sample

Processor가 반환하는 `image_pixel_values`, `image_sizes`,
`image_attention_mask` 등을 이름별로 하드코딩하지 않고 batch dict 전체를
모델에 전달합니다. 패딩은 `attention_mask`로만 가려 실제 EOS 토큰을
실수로 제거하지 않습니다.


In [14]:
def extract_pil_images(messages):
    return [
        block["image"]
        for message in messages
        for block in message["content"]
        if block.get("type") == "image"
    ]


class Phi4OrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)
        full_text = apply_phi_chat_template(
            messages,
            tokenize=False,
        )
        prompt_text = apply_phi_chat_template(
            messages[:1],
            tokenize=False,
            add_generation_prompt=True,
        )
        return {
            "id": row["Id"],
            "full_text": full_text,
            "prompt_text": prompt_text,
            "images": extract_pil_images(messages),
            "target_text": target_text,
        }


def collate_fn(samples):
    full_texts = [sample["full_text"] for sample in samples]
    prompt_texts = [sample["prompt_text"] for sample in samples]
    pil_images = [
        image
        for sample in samples
        for image in sample["images"]
    ]
    assert len(pil_images) == 4 * len(samples)

    try:
        full_enc = processor(
            text=full_texts,
            images=pil_images,
            padding="longest",
            truncation=False,
            dynamic_hd=PHI_DYNAMIC_HD,
            return_tensors="pt",
        )
        prompt_enc = processor(
            text=prompt_texts,
            images=pil_images,
            padding="longest",
            truncation=False,
            dynamic_hd=PHI_DYNAMIC_HD,
            return_tensors="pt",
        )
    finally:
        for image in pil_images:
            image.close()

    labels = full_enc["input_ids"].clone()
    prompt_lengths = prompt_enc["attention_mask"].sum(dim=1)
    full_lengths = full_enc["attention_mask"].sum(dim=1)
    assert torch.all(prompt_lengths < full_lengths), (
        prompt_lengths,
        full_lengths,
    )

    for row_index, prompt_length in enumerate(prompt_lengths.tolist()):
        assert torch.equal(
            full_enc["input_ids"][row_index, :prompt_length],
            prompt_enc["input_ids"][row_index, :prompt_length],
        ), f"full/prompt token prefix 불일치: row={row_index}"
        labels[row_index, :prompt_length] = -100
    labels[full_enc["attention_mask"] == 0] = -100
    assert torch.all((labels != -100).sum(dim=1) > 0)

    batch = dict(full_enc)
    batch["labels"] = labels
    return batch


In [15]:
import bitsandbytes as bnb
from safetensors.torch import save_file

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tr_df = train[train["split"] == "train"].reset_index(drop=True)
va_df = train[train["split"] == "validation"].reset_index(drop=True)
print(f"Train: {len(tr_df)}, Val: {len(va_df)}")

tr_ds = Phi4OrderingDataset(tr_df, f"{DATA_DIR}/train", processor)
va_ds = Phi4OrderingDataset(va_df, f"{DATA_DIR}/train", processor)

TRAIN_NUM_WORKERS = min(8, max(2, (os.cpu_count() or 8) // 4))
loader_generator = torch.Generator().manual_seed(SEED)
common_loader_kwargs = {
    "collate_fn": collate_fn,
    "num_workers": TRAIN_NUM_WORKERS,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 2,
}
tr_loader = DataLoader(
    tr_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    **common_loader_kwargs,
)
va_loader = DataLoader(
    va_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    **common_loader_kwargs,
)
print(
    f"DataLoader workers={TRAIN_NUM_WORKERS}, "
    f"prefetched batches per worker=2"
)

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]
optimizer = bnb.optim.AdamW8bit(
    trainable_parameters,
    lr=2e-4,
    betas=(0.9, 0.95),
    weight_decay=0.01,
)

MAX_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
MAX_GRAD_NORM = 1.0

BEST_ADAPTER_PATH = (
    Path(OUTPUT_DIR) / "phi4_vision_lora_best.safetensors"
)
HISTORY_PATH = Path(OUTPUT_DIR) / "phi4_vision_lora_history.json"
RUN_CONFIG_PATH = Path(OUTPUT_DIR) / "phi4_vision_lora_run_config.json"


def save_trainable_state(model, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    state = {
        name: parameter.detach().to(torch.bfloat16).cpu().contiguous()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }
    assert set(state) == set(TRAINABLE_PARAM_NAMES)
    temporary_path = Path(str(path) + ".tmp")
    save_file(
        state,
        str(temporary_path),
        metadata={
            "model_name": MODEL_NAME,
            "model_revision": MODEL_REVISION,
            "adapter_name": "vision",
            "adapter_subfolder": VISION_ADAPTER_SUBFOLDER,
        },
    )
    os.replace(temporary_path, path)
    print(f"LoRA 가중치 저장 완료: {path}")


def save_json_atomic(payload, path):
    path = Path(path)
    temporary_path = Path(str(path) + ".tmp")
    with open(temporary_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
    os.replace(temporary_path, path)


run_config = {
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "adapter_name": "vision",
    "adapter_subfolder": VISION_ADAPTER_SUBFOLDER,
    "train_rows": len(tr_df),
    "validation_rows": len(va_df),
    "seed": SEED,
    "dynamic_hd": PHI_DYNAMIC_HD,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "learning_rate": 2e-4,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "gpu_name": GPU_NAME,
    "gpu_vram_gib": GPU_VRAM_GB,
    "trainable_parameter_count": trainable_params,
    "package_versions": PACKAGE_VERSIONS,
}
save_json_atomic(run_config, RUN_CONFIG_PATH)

# Prompt order and label masking check.
preview_sample = tr_ds[0]
print("===== rendered full_text (first train sample) =====")
print(preview_sample["full_text"])
image_token = processor.image_token
assert preview_sample["full_text"].count(image_token) == 4
for image_number in range(1, 5):
    assert f"Image {image_number}" in preview_sample["full_text"]

preview_batch = collate_fn([preview_sample])
supervised_ids = preview_batch["labels"][0][
    preview_batch["labels"][0] != -100
]
supervised_text = processor.tokenizer.decode(
    supervised_ids,
    skip_special_tokens=False,
)
print("===== supervised label text =====")
print(supervised_text)
assert preview_sample["target_text"] in supervised_text
print(
    "Batch keys:",
    {
        key: tuple(value.shape)
        for key, value in preview_batch.items()
        if torch.is_tensor(value)
    },
)
del preview_batch, preview_sample, supervised_ids
gc.collect()

# 실제 설정으로 forward/backward 1회. optimizer.step은 하지 않는다.
def move_batch_to_model(batch):
    return {
        key: value.to(model.device, non_blocking=True)
        if torch.is_tensor(value)
        else value
        for key, value in batch.items()
    }


model.train()
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
dry_samples = [
    tr_ds[index]
    for index in range(TRAIN_BATCH_SIZE)
]
dry_batch = move_batch_to_model(collate_fn(dry_samples))
dry_outputs = model(**dry_batch)
dry_loss = dry_outputs.loss
assert torch.isfinite(dry_loss), dry_loss
dry_loss.backward()
gradient_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and parameter.grad is not None
]
assert gradient_names, "vision LoRA에 gradient가 생성되지 않았습니다."
dry_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print(
    f"Dry run 통과: loss={dry_loss.item():.4f}, "
    f"gradient tensors={len(gradient_names)}, "
    f"peak_allocated={dry_peak_gib:.1f} GiB"
)
optimizer.zero_grad(set_to_none=True)
del dry_samples, dry_batch, dry_outputs, dry_loss
gc.collect()
torch.cuda.empty_cache()
DRY_RUN_PASSED = True
print(
    "위 loss/VRAM을 확인하세요. OOM 여유가 작으면 GPU 설정 셀의 "
    "TRAIN_BATCH_SIZE를 낮추고 이 셀을 다시 실행하세요."
)


Train: 7628, Val: 1907
DataLoader workers=8, prefetched batches per worker=2
===== rendered full_text (first train sample) =====
<|user|><|endoftext10|>
Image 1
<|endoftext10|>
Image 2
<|endoftext10|>
Image 3
<|endoftext10|>
Image 4
Thinking about the sentence: "The camera pans left to reveal two people kneeling beside a fishing hole on the ice, then shifts from a frontal to a rear view as a skier moves forward and jumps onto a rail."
This caption describes a single continuous action or a brief moment. Focus on subtle visual cues such as camera movement direction, object position changes, hand/body motion, and lighting changes to infer the temporal order.
Look at the 4 images above labeled Image 1 to Image 4. Determine the correct chronological order of these images to match the sentence. Provide the answer ONLY as a Python list of integers. Example: [1, 2, 3, 4]<|end|><|assistant|>[1, 2, 3, 4]<|end|><|endoftext|>
===== supervised label text =====
[1, 2, 3, 4]<|end|><|endoftext|>
Batch

### 7. Confirm the dry run and start training

위 셀의 유한 loss, LoRA gradient 수, peak VRAM을 확인한 뒤에만
`START_LORA_TRAINING=True`로 바꾸세요. 이 셀이 실제 optimizer update를
시작하는 유일한 위치입니다.


In [ ]:
START_LORA_TRAINING = True
assert START_LORA_TRAINING is True, (
    "Dry run의 loss와 peak VRAM을 확인한 뒤 "
    "START_LORA_TRAINING=True로 바꾸세요."
)
assert DRY_RUN_PASSED is True


def evaluate_validation_loss(model, loader):
    model.eval()
    loss_sum = 0.0
    sample_count = 0
    with torch.inference_mode():
        for batch in tqdm(
            loader,
            desc="Validation loss",
            leave=False,
        ):
            batch = move_batch_to_model(batch)
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"비정상 validation loss: {loss.item()}"
                )
            batch_size = batch["input_ids"].shape[0]
            loss_sum += loss.item() * batch_size
            sample_count += batch_size
    return loss_sum / sample_count


best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
training_history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    sample_count = 0

    for step, batch in enumerate(
        tqdm(
            tr_loader,
            desc=f"Epoch {epoch + 1}/{MAX_EPOCHS}",
        )
    ):
        batch = move_batch_to_model(batch)
        outputs = model(**batch)
        raw_loss = outputs.loss
        if not torch.isfinite(raw_loss):
            raise RuntimeError(
                f"비정상 loss 감지: step={step}, "
                f"loss={raw_loss.item()}"
            )
        (raw_loss / GRAD_ACCUM_STEPS).backward()

        is_update_step = (
            (step + 1) % GRAD_ACCUM_STEPS == 0
            or step + 1 == len(tr_loader)
        )
        if is_update_step:
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=MAX_GRAD_NORM,
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        batch_size = batch["input_ids"].shape[0]
        loss_sum += raw_loss.item() * batch_size
        sample_count += batch_size
        if step % 100 == 0:
            peak_gib = (
                torch.cuda.max_memory_allocated() / 1024**3
            )
            print(
                f" step {step}: loss={raw_loss.item():.4f}, "
                f"peak_alloc={peak_gib:.1f} GiB"
            )

    train_loss = loss_sum / sample_count
    val_loss = evaluate_validation_loss(model, va_loader)
    training_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
        }
    )
    save_json_atomic(training_history, HISTORY_PATH)
    print(
        f"Epoch {epoch + 1}: train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}"
    )

    if val_loss < best_val_loss - EARLY_STOPPING_MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        save_trainable_state(model, BEST_ADAPTER_PATH)
        print(
            f"새 best adapter: epoch={best_epoch}, "
            f"val_loss={best_val_loss:.4f}"
        )
    else:
        epochs_without_improvement += 1
        print(
            f"개선 없음: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )
        if (
            epochs_without_improvement
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping at epoch {epoch + 1}; "
                f"best epoch={best_epoch}"
            )
            break

print(
    f"Best epoch={best_epoch}, "
    f"best validation loss={best_val_loss:.4f}"
)
print(f"Best adapter path: {BEST_ADAPTER_PATH}")
print(
    "최종 peak allocated: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.1f} GiB"
)


Epoch 1/3:   0%|          | 0/1907 [00:00<?, ?it/s]

 step 0: loss=0.1428, peak_alloc=49.3 GiB
 step 100: loss=0.2255, peak_alloc=51.9 GiB
 step 200: loss=0.2402, peak_alloc=52.0 GiB


### 7B. Recover an interrupted epoch 2 from the saved epoch 1 best

이 셀은 epoch 1 완료 후 epoch 2 도중 중단된 경우에만 사용합니다. 중간까지
업데이트된 epoch 2 상태와 optimizer를 버리고, 저장된 epoch 1 best LoRA를
복원한 뒤 새 optimizer로 epoch 2 전체를 한 번 학습합니다. optimizer state는
checkpoint에 없으므로 weight-only warm restart이며, 완료 이력이 이미 `[1, 2]`
이상이면 재실행하지 않습니다.


In [16]:
import bitsandbytes as bnb
from safetensors.torch import load_file

RECOVER_INTERRUPTED_EPOCH_2 = True
assert RECOVER_INTERRUPTED_EPOCH_2 is True
assert PHI4_LORA_GPU_CONFIRMED is True
assert BEST_ADAPTER_PATH.is_file(), BEST_ADAPTER_PATH
assert HISTORY_PATH.is_file(), HISTORY_PATH

with open(HISTORY_PATH, "r", encoding="utf-8") as handle:
    completed_history = json.load(handle)
assert isinstance(completed_history, list) and completed_history
completed_epochs = [int(record["epoch"]) for record in completed_history]

if completed_epochs != [1]:
    print(
        "Epoch-2 recovery skipped: expected completed history [1], "
        f"found {completed_epochs}."
    )
else:
    previous_best_record = min(
        completed_history,
        key=lambda record: float(record["val_loss"]),
    )
    previous_best_epoch = int(previous_best_record["epoch"])
    previous_best_loss = float(previous_best_record["val_loss"])
    assert previous_best_epoch == 1

    # Drop the interrupted graph, gradients, optimizer state, and workers.
    for parameter in model.parameters():
        parameter.grad = None
    for variable_name in (
        "optimizer",
        "trainable_parameters",
        "outputs",
        "raw_loss",
        "batch",
        "tr_loader",
        "va_loader",
        "tr_ds",
        "va_ds",
    ):
        globals().pop(variable_name, None)
    gc.collect()
    torch.cuda.empty_cache()

    # Restore every trainable tensor from the completed epoch-1 checkpoint.
    restored_state = load_file(str(BEST_ADAPTER_PATH))
    expected_trainable_names = set(TRAINABLE_PARAM_NAMES)
    assert set(restored_state) == expected_trainable_names
    incompatible = model.load_state_dict(restored_state, strict=False)
    assert not incompatible.unexpected_keys, incompatible.unexpected_keys
    del restored_state

    resumed_trainable_names = []
    for name, parameter in model.named_parameters():
        should_train = name in expected_trainable_names
        parameter.requires_grad_(should_train)
        if should_train:
            parameter.data = parameter.data.to(torch.float32)
            resumed_trainable_names.append(name)
    assert set(resumed_trainable_names) == expected_trainable_names
    assert model.active_adapters() == ["vision"]
    assert model.device.type == "cuda", model.device

    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    if not hasattr(model, "_require_grads_hook"):
        model.enable_input_require_grads()
    model.config.use_cache = False
    processor.tokenizer.padding_side = "right"

    tr_ds = Phi4OrderingDataset(tr_df, f"{DATA_DIR}/train", processor)
    va_ds = Phi4OrderingDataset(va_df, f"{DATA_DIR}/train", processor)
    resume_loader_generator = torch.Generator().manual_seed(SEED + 1)
    resume_loader_kwargs = {
        "collate_fn": collate_fn,
        "num_workers": TRAIN_NUM_WORKERS,
        "pin_memory": True,
        "persistent_workers": True,
        "prefetch_factor": 2,
    }
    tr_loader = DataLoader(
        tr_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=resume_loader_generator,
        **resume_loader_kwargs,
    )
    va_loader = DataLoader(
        va_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        **resume_loader_kwargs,
    )

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]
    optimizer = bnb.optim.AdamW8bit(
        trainable_parameters,
        lr=2e-4,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    def evaluate_recovered_epoch(model, loader):
        model.eval()
        validation_loss_sum = 0.0
        validation_sample_count = 0
        with torch.inference_mode():
            for validation_batch in tqdm(
                loader,
                desc="Recovered epoch 2 validation",
                leave=False,
            ):
                validation_batch = move_batch_to_model(validation_batch)
                validation_loss = model(**validation_batch).loss
                assert torch.isfinite(validation_loss), validation_loss
                validation_batch_size = validation_batch["input_ids"].shape[0]
                validation_loss_sum += (
                    validation_loss.item() * validation_batch_size
                )
                validation_sample_count += validation_batch_size
        return validation_loss_sum / validation_sample_count

    resumed_epoch = 2
    model.train()
    optimizer.zero_grad(set_to_none=True)
    resumed_loss_sum = 0.0
    resumed_sample_count = 0
    torch.cuda.reset_peak_memory_stats()

    for resume_step, resume_batch in enumerate(
        tqdm(tr_loader, desc="Recovered epoch 2/2")
    ):
        resume_batch = move_batch_to_model(resume_batch)
        resume_outputs = model(**resume_batch)
        resume_raw_loss = resume_outputs.loss
        if not torch.isfinite(resume_raw_loss):
            raise RuntimeError(
                f"Non-finite recovered loss: step={resume_step}, "
                f"loss={resume_raw_loss.item()}"
            )
        (resume_raw_loss / GRAD_ACCUM_STEPS).backward()

        resume_is_update_step = (
            (resume_step + 1) % GRAD_ACCUM_STEPS == 0
            or resume_step + 1 == len(tr_loader)
        )
        if resume_is_update_step:
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=MAX_GRAD_NORM,
            )
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        resumed_batch_size = resume_batch["input_ids"].shape[0]
        resumed_loss_sum += resume_raw_loss.item() * resumed_batch_size
        resumed_sample_count += resumed_batch_size
        if resume_step % 100 == 0:
            resumed_peak_gib = (
                torch.cuda.max_memory_allocated() / 1024**3
            )
            print(
                f" resume step {resume_step}: "
                f"loss={resume_raw_loss.item():.4f}, "
                f"peak_alloc={resumed_peak_gib:.1f} GiB"
            )

    resumed_train_loss = resumed_loss_sum / resumed_sample_count
    resumed_val_loss = evaluate_recovered_epoch(model, va_loader)
    resumed_is_best = (
        resumed_val_loss
        < previous_best_loss - EARLY_STOPPING_MIN_DELTA
    )

    if resumed_is_best:
        save_trainable_state(model, BEST_ADAPTER_PATH)
        best_epoch = resumed_epoch
        best_val_loss = resumed_val_loss
    else:
        best_epoch = previous_best_epoch
        best_val_loss = previous_best_loss

    completed_history.append(
        {
            "epoch": resumed_epoch,
            "train_loss": resumed_train_loss,
            "val_loss": resumed_val_loss,
            "resumed_from_best_epoch": previous_best_epoch,
            "optimizer_state_restored": False,
        }
    )
    training_history = completed_history
    save_json_atomic(training_history, HISTORY_PATH)

    if RUN_CONFIG_PATH.is_file():
        with open(RUN_CONFIG_PATH, "r", encoding="utf-8") as handle:
            recovery_run_config = json.load(handle)
    else:
        recovery_run_config = {}
    recovery_run_config.update(
        {
            "completed_epochs": 2,
            "training_stop_epoch": 2,
            "recovery_mode": "epoch1_best_weight_only_then_one_epoch",
            "recovery_optimizer_state_restored": False,
            "best_epoch": best_epoch,
            "best_validation_loss": best_val_loss,
        }
    )
    save_json_atomic(recovery_run_config, RUN_CONFIG_PATH)

    # Release the final training graph before section 8 reloads the model.
    for variable_name in (
        "resume_outputs",
        "resume_raw_loss",
        "resume_batch",
    ):
        globals().pop(variable_name, None)
    gc.collect()
    torch.cuda.empty_cache()

    print(
        f"Recovered epoch 2 complete: train_loss={resumed_train_loss:.4f}, "
        f"val_loss={resumed_val_loss:.4f}"
    )
    print(
        f"Best epoch after two completed epochs={best_epoch}, "
        f"best validation loss={best_val_loss:.4f}"
    )
    print("Run section 8 next; do not rerun the original training cell.")


Recovered epoch 2/2:   0%|          | 0/1907 [00:00<?, ?it/s]

 resume step 0: loss=0.2699, peak_alloc=49.3 GiB
 resume step 100: loss=0.2570, peak_alloc=52.0 GiB
 resume step 200: loss=0.2553, peak_alloc=52.0 GiB
 resume step 300: loss=0.2357, peak_alloc=52.0 GiB
 resume step 400: loss=0.2241, peak_alloc=52.0 GiB
 resume step 500: loss=0.2283, peak_alloc=52.0 GiB
 resume step 600: loss=0.2700, peak_alloc=52.0 GiB
 resume step 700: loss=0.2191, peak_alloc=52.0 GiB
 resume step 800: loss=0.1741, peak_alloc=52.0 GiB
 resume step 900: loss=0.2437, peak_alloc=52.0 GiB
 resume step 1000: loss=0.2860, peak_alloc=52.0 GiB
 resume step 1100: loss=0.2363, peak_alloc=52.0 GiB
 resume step 1200: loss=0.2221, peak_alloc=52.0 GiB
 resume step 1300: loss=0.2819, peak_alloc=52.0 GiB
 resume step 1400: loss=0.2465, peak_alloc=52.0 GiB
 resume step 1500: loss=0.2344, peak_alloc=52.0 GiB
 resume step 1600: loss=0.2431, peak_alloc=52.0 GiB
 resume step 1700: loss=0.2326, peak_alloc=52.0 GiB
 resume step 1800: loss=0.2931, peak_alloc=52.0 GiB
 resume step 1900: loss=

Recovered epoch 2 validation:   0%|          | 0/477 [00:00<?, ?it/s]

Recovered epoch 2 complete: train_loss=0.2631, val_loss=0.2625
Best epoch after two completed epochs=1, best validation loss=0.2484
Run section 8 next; do not rerun the original training cell.


### 8. Reload only the saved LoRA subset for inference

새 base 모델과 공식 `vision-lora`를 다시 로드한 뒤, 학습된 A/B tensor
subset을 이름 그대로 덮어씁니다.


In [17]:
from safetensors.torch import load_file

# 전역 notebook 변수에 남은 graph/parameter/worker 참조까지 제거한다.
for variable_name in (
    "optimizer",
    "trainable_parameters",
    "outputs",
    "raw_loss",
    "batch",
    "resume_outputs",
    "resume_raw_loss",
    "resume_batch",
    "tr_loader",
    "va_loader",
    "tr_ds",
    "va_ds",
    "model",
):
    globals().pop(variable_name, None)
gc.collect()
torch.cuda.empty_cache()

assert BEST_ADAPTER_PATH.is_file(), BEST_ADAPTER_PATH
processor, model, adapter_info = load_phi4_with_vision_adapter(
    is_trainable=False
)

tuned_state = load_file(str(BEST_ADAPTER_PATH))
model_state_names = set(model.state_dict())
absent_names = sorted(set(tuned_state) - model_state_names)
assert not absent_names, absent_names[:20]

incompatible = model.load_state_dict(tuned_state, strict=False)
assert not incompatible.unexpected_keys, incompatible.unexpected_keys
assert model.active_adapters() == ["vision"]

processor.tokenizer.padding_side = "left"
model.config.use_cache = True
model.eval()

first_tuned_name = next(iter(tuned_state))
restored_mean = (
    model.state_dict()[first_tuned_name].float().abs().mean().item()
)
print(f"Reloaded tuned vision LoRA: {BEST_ADAPTER_PATH}")
print(
    f"첫 tuned tensor 확인: {first_tuned_name}, "
    f"abs_mean={restored_mean:.6g}"
)
del tuned_state


Native Phi processor ready: Phi4MultimodalProcessor Phi4MultimodalImageProcessor Phi4MultimodalFeatureExtractor


Loading weights:   0%|          | 0/1535 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

[transformers] Phi4MultimodalForCausalLM LOAD REPORT from: microsoft/Phi-4-multimodal-instruct
Key                                                                      | Status  | 
-------------------------------------------------------------------------+---------+-
model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.bias   | MISSING | 
model.embed_tokens_extend.audio_embed.down_proj_for_vision_speech.weight | MISSING | 
model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.bias     | MISSING | 
model.embed_tokens_extend.audio_embed.up_proj_for_vision_speech.weight   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reloaded tuned vision LoRA: /content/drive/MyDrive/snu ai/snu_phi4_multimodal_outputs/phi4_vision_lora_best.safetensors
첫 tuned tensor 확인: model.layers.0.mlp.down_proj.lora_A.vision.weight, abs_mean=0.00937483


In [18]:
from scipy.stats import kendalltau


def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [
        kendalltau(p, t)[0]
        if not np.isnan(kendalltau(p, t)[0])
        else 1.0
        for p, t in zip(pred, target)
    ]
    return {
        "exact_match": exact_match,
        "position_acc": position_acc,
        "kendall_tau": np.mean(taus),
    }


def build_generation_inputs(batch_rows, image_dir):
    batch_messages = [
        get_prompt_message(row, image_dir)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        apply_phi_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in batch_messages
    ]
    pil_images = [
        image
        for messages in batch_messages
        for image in extract_pil_images(messages)
    ]
    assert len(pil_images) == 4 * len(batch_rows)
    try:
        inputs = processor(
            text=batch_texts,
            images=pil_images,
            padding="longest",
            dynamic_hd=PHI_DYNAMIC_HD,
            return_tensors="pt",
        )
    finally:
        for image in pil_images:
            image.close()
    return inputs.to(model.device)


processor.tokenizer.padding_side = "left"
model.eval()
results = []

for start_idx in tqdm(
    range(0, len(va_df), INFERENCE_BATCH_SIZE)
):
    batch_rows = va_df.iloc[
        start_idx:start_idx + INFERENCE_BATCH_SIZE
    ]
    inputs = build_generation_inputs(
        batch_rows,
        f"{DATA_DIR}/train",
    )
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )

    prompt_width = inputs["input_ids"].shape[1]
    generated_only = generated_ids[:, prompt_width:]
    output_texts = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    for (_, row), output_text in zip(
        batch_rows.iterrows(),
        output_texts,
    ):
        pred, parse_ok = parse_model_output(output_text)
        results.append(
            {
                "Id": row["Id"],
                "pred": pred,
                "parse_ok": parse_ok,
                "raw_output": output_text,
            }
        )

assert len(results) == len(va_df)
assert [result["Id"] for result in results] == va_df["Id"].tolist()

pred_array = np.array([result["pred"] for result in results]) - 1
target_array = (
    np.array([ast.literal_eval(answer) for answer in va_df["Answer"]])
    - 1
)

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df["No_ordering"].astype(bool).values
shuffled = compute_metrics(
    pred_array[~no_order_mask],
    target_array[~no_order_mask],
)
parse_failure_rate = 1 - np.mean(
    [result["parse_ok"] for result in results]
)

print(
    "[Phi-4-multimodal + built-in vision LoRA, "
    f"MD5-group-stratified 80:20 val, n={len(va_df)}]"
)
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={parse_failure_rate:.1%}")

validation_predictions_df = pd.DataFrame(
    {
        "Id": va_df["Id"],
        "Answer": va_df["Answer"],
        "Prediction": [str(result["pred"]) for result in results],
        "Correct": (pred_array == target_array).all(axis=1),
        "ParseOk": [result["parse_ok"] for result in results],
        "No_ordering": va_df["No_ordering"],
        "RawOutput": [
            result["raw_output"] for result in results
        ],
    }
)
validation_csv_path = (
    Path(OUTPUT_DIR)
    / "validation_predictions_phi4_multimodal_lora.csv"
)
validation_predictions_df.to_csv(
    validation_csv_path,
    index=False,
    encoding="utf-8-sig",
)
print(f"Validation CSV 저장 완료: {validation_csv_path}")
display(validation_predictions_df.head())


  0%|          | 0/477 [00:00<?, ?it/s]

[Phi-4-multimodal + built-in vision LoRA, MD5-group-stratified 80:20 val, n=1907]
  전체 exact=0.1552
  셔플됨 exact=0.0000
  파싱 실패율=0.2%
Validation CSV 저장 완료: /content/drive/MyDrive/snu ai/snu_phi4_multimodal_outputs/validation_predictions_phi4_multimodal_lora.csv


,Id,Answer,Prediction,Correct,ParseOk,No_ordering,RawOutput
0,2vqGOF,"[1, 2, 3, 4]","[1, 2, 3, 4]",True,True,True,"[1, 2, 3, 4]"
1,yMg9ew,"[1, 2, 3, 4]","[1, 2, 3, 4]",True,True,True,"[1, 2, 3, 4]"
2,FIE67H,"[2, 4, 1, 3]","[1, 2, 3, 4]",False,True,False,"[1, 2, 3, 4]"
3,qHYE4w,"[1, 4, 2, 3]","[1, 2, 3, 4]",False,True,False,"[1, 2, 3, 4]"
4,mTu2ny,"[3, 2, 4, 1]","[1, 2, 3, 4]",False,True,False,"[1, 2, 3, 4]"


### 9. Optional test inference and submission

아래 셀은 validation을 마친 뒤에만 실행합니다. Test는 학습, split 생성,
early stopping, 모델 선택에 사용하지 않습니다.


In [19]:
test_path = Path(DATA_DIR) / "test.csv"
test_image_dir = Path(DATA_DIR) / "test"
sample_submission_path = Path(DATA_DIR) / "sample_submission.csv"
assert test_path.is_file(), test_path
assert test_image_dir.is_dir(), test_image_dir
assert sample_submission_path.is_file(), sample_submission_path

test = pd.read_csv(test_path)
print(f"Test samples: {len(test)}")


Test samples: 819


In [20]:
torch.cuda.empty_cache()
gc.collect()

processor.tokenizer.padding_side = "left"
TEST_IMAGE_DIR = f"{DATA_DIR}/test"

# 같은 런타임에서 이 셀을 재실행하면 완료된 Id는 보존한다.
if "test_results" not in globals():
    test_results = []

processed_ids = {result["Id"] for result in test_results}
remaining_test = test[
    ~test["Id"].isin(processed_ids)
].reset_index(drop=True)
print(f"Remaining: {len(remaining_test)} / {len(test)}")


def generate_rows(batch_rows, image_dir):
    inputs = build_generation_inputs(batch_rows, image_dir)
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
        )
    prompt_width = inputs["input_ids"].shape[1]
    return processor.batch_decode(
        generated_ids[:, prompt_width:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


model.eval()
for start_idx in tqdm(
    range(0, len(remaining_test), INFERENCE_BATCH_SIZE)
):
    batch_rows = remaining_test.iloc[
        start_idx:start_idx + INFERENCE_BATCH_SIZE
    ]
    try:
        output_texts = generate_rows(
            batch_rows,
            TEST_IMAGE_DIR,
        )
    except torch.cuda.OutOfMemoryError:
        # Multimodal tensor를 자르지 않고 각 4-image sample을 다시 만든다.
        torch.cuda.empty_cache()
        output_texts = []
        for row_index in range(len(batch_rows)):
            one_row = batch_rows.iloc[row_index:row_index + 1]
            output_texts.extend(
                generate_rows(one_row, TEST_IMAGE_DIR)
            )

    for (_, row), output_text in zip(
        batch_rows.iterrows(),
        output_texts,
    ):
        pred, parse_ok = parse_model_output(output_text)
        test_results.append(
            {
                "Id": row["Id"],
                "pred": pred,
                "parse_ok": parse_ok,
                "raw_output": output_text,
            }
        )

print(f"Completed: {len(test_results)} / {len(test)}")
print(
    "Parse failure rate: "
    f"{1 - np.mean([r['parse_ok'] for r in test_results]):.1%}"
)


Remaining: 819 / 819


  0%|          | 0/205 [00:00<?, ?it/s]

Completed: 819 / 819
Parse failure rate: 0.5%


In [21]:
print(f"지금까지 처리된 개수: {len(test_results)}")


지금까지 처리된 개수: 819


In [22]:
sample_submission = pd.read_csv(
    f"{DATA_DIR}/sample_submission.csv"
)
prediction_by_id = {
    result["Id"]: str(result["pred"])
    for result in test_results
}
submission_df = sample_submission[["Id"]].copy()
submission_df["Answer"] = submission_df["Id"].map(prediction_by_id)

assert list(submission_df.columns) == ["Id", "Answer"]
assert len(submission_df) == len(sample_submission)
assert submission_df["Answer"].notna().all(), "미완료 test Id가 있습니다."

valid_answers = {
    str(list(order))
    for order in permutations([1, 2, 3, 4])
}
assert set(submission_df["Answer"]).issubset(valid_answers)

submit_path = (
    Path(OUTPUT_DIR) / "submission_phi4_multimodal_lora.csv"
)
submission_df.to_csv(submit_path, index=False)

saved_submission = pd.read_csv(submit_path)
assert saved_submission["Id"].tolist() == sample_submission["Id"].tolist()
assert saved_submission["Answer"].notna().all()

print(f"저장 완료: {submit_path}")
print("형식 검증 통과 — 제출 가능합니다.")
display(submission_df.head())


저장 완료: /content/drive/MyDrive/snu ai/snu_phi4_multimodal_outputs/submission_phi4_multimodal_lora.csv
형식 검증 통과 — 제출 가능합니다.


,Id,Answer
0,WI8o3F,"[1, 2, 3, 4]"
1,Ae7zLm,"[1, 2, 3, 4]"
2,4QSIII,"[1, 2, 3, 4]"
3,k7h4gt,"[1, 2, 3, 4]"
4,ZtTwCx,"[1, 2, 3, 4]"


In [23]:
pred_strs = [str(result["pred"]) for result in test_results]
print("전체 예측 분포 상위 5개:", Counter(pred_strs).most_common(5))


전체 예측 분포 상위 5개: [('[1, 2, 3, 4]', 819)]


## Next Steps

1. GPU 확인 gate를 통과시킵니다.
2. 모델/adapter 이름, 첫 prompt, supervised label, dry-run loss와 VRAM을 확인합니다.
3. 두 번째 gate에서만 실제 학습을 시작합니다.
4. validation Exact Match와 parse-failure rate가 나온 뒤 `MODELS.md`의
   EXP-013에 수치, 시간, peak VRAM, 다음 결정을 기록합니다.
